# 3.3 Your turn: events, not autocorrelations

[03.2](03.2-statistics-of-time.ipynb) found a cycle nobody was told about — in sunspots.
Your chat is not sunspots. A chat rarely hides an unknown rhythm: if the daily and weekly
cycles show nothing more than "people sleep at night and dont work in the weekend", then 
your plots are not really showing anything of value.
Point an autocorrelation at it and you will probably find lag 7, already knowing it was there —
and then go hunting for something, anything, in the other lags. The risk is that this is **finding a
problem for your solution**: the tool came first and the question got invented to fit it.

This notebook runs the other direction, and the order is the whole exercise:

1. **Name an event you already know happened.** A holiday, a birthday, exam week, a trip,
   someone joining or leaving the group, a move, covid, a fight.
2. **Write down what impact it *should* have left — before looking.** More messages or
   fewer? Different people? Different hours? Longer or shorter messages, more emoji,
   another language?
3. **Then build the one plot that would show it — or fail to.** The tools are all from
   this lesson: reindexed daily counts, a rolling average, a date line, and 03.1's move —
   subtract the baseline, read what is left.

A prediction written down before looking is the difference between checking a hypothesis
and rationalising a squiggle.

In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    FlagDates,
    Pipeline,
    RegexFeature,
    RollingAvg,
    SortValues,
    SubtractBaseline,
    TimeFeatures,
    TransformBase,
)
from goad_toolkit.visualizer import (
    HorizontalLine,
    LinePlot,
    PlotSettings,
    VerticalDate,
)

from scripts.plots import BarPlot, EventMarker, RiseDeclinePlot
from wa_analyzer.data import load_own_chat

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])

## 3.3.1 Step one: events you know about

Fill the dict below from memory, not from the data — that is the point. If nothing comes
to mind yet, the fallback picks your busiest day ever and asks you the question in
reverse: *you* tell *it* what happened there. (Careful with the reverse direction: a spike
you then explain is a story, not a tested prediction — use it to get started, not to
conclude.)

In [ ]:
my_events = {
    # >>> Your turn: events you KNOW about, from memory -- "label": "YYYY-MM-DD" <<<
    "Positive pregnancy test": "2024-09-01",
    "Benji is born": "2025-05-11"

}

daily = own.set_index("timestamp").resample("D").size().rename("messages").reset_index()

if not my_events:
    busiest = daily.loc[daily.messages.idxmax(), "timestamp"]
    my_events = {"busiest day ever -- what happened here?": str(busiest.date())}

for label, day in my_events.items():
    print(f"{day}  {label}")

## 3.3.2 Step two: what would the impact look like?

Write the expectation down first, next to the event. Different impacts live in different
plots — pick the plot from the prediction, never the other way around:

| if the event changed... | you would expect | the plot that would show it |
|---|---|---|
| **how much** | a spike or a dip in messages per day | daily counts + rolling average + a date line |
| **when** | the day's hourly shape shifts | hour shares around the event minus the ordinary shape — 03.1's residual move |
| **who** | someone appears, disappears, or takes over | messages per author, before vs after |
| **how** | length, emoji, links, questions, another language | a `RegexFeature` per marker (lesson 1), compared before vs after |

Two honesty checks before you trust anything you find. **Sample size**: one birthday is
one day — a difference between one day and the baseline can always be noise, and saying
so is a finding. **Shares, not counts**, whenever the volumes differ: a loud week wins
every raw comparison for boring reasons; 03.1 normalised each day to its own shares for
exactly this reason.

## 3.3.3 Worked example: the volume plot

Daily counts, a 7-day rolling average (`RollingAvg`, the pipeline form of the
`.rolling()` the flights showcase used — and the same warning: a wider window erases
shorter events), and one
`VerticalDate` per event. `resample("D")` already emits every calendar day, quiet days
as zero — the honest-gap lesson from 03.1, handled at the counting step instead of
patched in afterwards.

In [ ]:
smoothed = Pipeline().add(
    RollingAvg, column="messages", window=7, rename=True
).apply(daily)

volume = PlotSettings(
    figsize=(12, 4),
    title="Messages per day, with the events I remember marked",
    xlabel="",
    ylabel="messages per day",
    xtick_rotation=45,
)
lines = LinePlot(volume)
fig, ax = lines.plot(data=daily, x="timestamp", y="messages", color="#cccccc",
                     label="daily")
lines.plot_on(LinePlot(volume), data=smoothed, x="timestamp",
              y="messages_rolling_avg", color="crimson", label="7-day average")
for label, day in my_events.items():
    lines.plot_on(VerticalDate(volume), date=day, label=label)
ax.legend()

Read it against your prediction, not for surprises: did the event you named move the
line the way you said it would? A spike you predicted is evidence. A spike you noticed
and then explained is a hypothesis for the *next* check, not a conclusion of this one.

## 3.3.4 Worked example: did the *shape* of the day change?

03.1's move, on your own data. Flag a window around one event (`FlagDates` matches whole
calendar days, so a timestamp column works as-is), build the hourly share inside the
window and the ordinary hourly share outside it, and let `SubtractBaseline` leave the
difference. Zero line means "this hour looked like any other day"; everything else is the
event's own signature.

In [ ]:
label, day = next(iter(my_events.items()))
event_day = pd.Timestamp(day)
window = pd.date_range(event_day - pd.Timedelta(days=3),
                       event_day + pd.Timedelta(days=3))

flagged = (
    Pipeline()
    .add(TimeFeatures, column="timestamp", features=["hour"])
    .add(FlagDates, column="timestamp", dates=window, feature="in_window")
    .apply(own)
)

baseline = (
    Pipeline()
    .add(Filter, expr="not in_window")
    .add(CountValues, column="hour", feature="share", normalize=True)
    .add(SortValues, column="hour")
    .apply(flagged)
)

event_shape = (
    Pipeline()
    .add(Filter, expr="in_window")
    .add(CountValues, column="hour", feature="share", normalize=True)
    .add(SortValues, column="hour")
    .add(SubtractBaseline, column="share", baseline=baseline, on="hour",
         feature="difference")
    .apply(flagged)
)

shape = PlotSettings(
    figsize=(9, 4),
    title=f"The week around '{label}', minus an ordinary day",
    xlabel="hour",
    ylabel="share of messages, event week − ordinary",
)
lines = LinePlot(shape)
fig, ax = lines.plot(data=event_shape, x="hour", y="difference", marker="o",
                     color="steelblue")
lines.plot_on(HorizontalLine(shape), y=0)

## 3.3.5 The *how* row: child and climbing around your events

The table in 3.3.2 has a row for events that change **how** you talk: which topics come up.
Notebook `roos_01.4` already built two topic categories with a `RegexFeature` each — `child`
and `climbing`. Here they are again, pointed at the events above instead of at the whole
relationship.

**Before you run the next cells, write your prediction here, in your own words** — which
topic should move at which event, in which direction, and roughly how much. Do not look at
the plot first.

> *My prediction:* (write it here)

In [ ]:
# The two categories from roos_01.4, patterns unchanged.
CATEGORY_PATTERNS: dict[str, str] = {
    "child": (
        r"(?i)("
        r"benji|b[ée]b[ée]|\bbaby\b|cr[eè]che|kinderdagverblijf|opvang|daycare|partou|basker|kinderdagopvang|"
        r"r[eé]cuperer|r[eé]cupes|prendre soin de|"
        r"\benfant\b|biberon|bottle[- ]?feeding|sucette|fopspeen|speentje|"
        r"pacifier|dummy|allaiter|donner les? seins?|borstvoeding|breastfeeding|t[ée]t[ée]e|"
        r"luiers?|diapers?|couches?|doekje|hydrophile|vitamines|hydrofiele doek|\bwipe\b|"
        r"kinderwagen|poussette|stroller|pram|la remorque|bike trailer|"
        r"consultatiebureau|kraamzorg|zwangerschap|grossesse|pregnan(cy|t)|"
        r"postnataal|zwanger|enceinte|"
        r"verloskundige|sage-femme|echo|midwife|kolven|[ée]chographie"
        r")"
    ),
    "climbing": (
        r"(?i)("
        r"escalad(er|e)|salle de grimpe|grimpe|grimper|\bclimb(ing)?\b|climbing gym|"
        r"klimmen|klimmuur( centraal)?|\bboulder\b|\bbloc\b|\bvoie\b|"
        r"arkose|mountain network|\bncc\b|neoliet|\b[4-8][abc]\+?\b|"
        r"national climbing center|pof|magnesium|prises|prise"
        r")"
    ),
}

topics = Pipeline()
for category, pattern in CATEGORY_PATTERNS.items():
    topics.add(
        RegexFeature,
        name=f"{category}_words",
        column="message",
        pattern=pattern,
        feature=f"n_{category}_words",
        mode="count",
    )
own_topics = topics.apply(own)

# One True/False column per topic: does this message mention it at all?
# The mean of that column over any group of messages is the topic's *share* in the group.
for category in CATEGORY_PATTERNS:
    own_topics[category] = own_topics[f"n_{category}_words"].gt(0).astype(float)

own_topics[list(CATEGORY_PATTERNS)].sum().astype(int).rename("messages mentioning it")

In [ ]:
FOCUS_COLORS = {"child": "#0072B2", "climbing": "#D55E00"}
SMOOTH_WEEKS = 4

# The two events the story is about, labelled the way they should read on the chart.
STORY_EVENTS = {
    "positive pregnancy test": my_events["Positive pregnancy test"],
    "baby is born": my_events["Benji is born"],
}


def topic_trend_plot(frame, colors, title):
    """Weekly share of messages per topic: grey raw weeks, coloured rolling average,
    a grey `EventMarker` (line + arrow + label) for each of STORY_EVENTS, and a caption
    under the plot saying how many messages the shares rest on."""
    # Share of each week's messages that mention the topic (shares, not counts: 03.3.2's rule).
    # A week with no messages at all stays NaN, so the line breaks instead of inventing a zero.
    weekly = (
        frame.set_index("timestamp")[list(colors)]
        .resample("W")
        .mean()
        .reset_index()
    )

    trend = Pipeline()
    for category in colors:
        trend.add(RollingAvg, column=category, window=SMOOTH_WEEKS, rename=True)
    weekly_trend = trend.apply(weekly)

    settings = PlotSettings(
        figsize=(12, 5),
        title=title,
        xlabel="",
        ylabel="share of weekly messages",
        xtick_rotation=45,
    )
    lines = LinePlot(settings)
    # The real weekly values recede to grey; the rolling average carries the colour.
    first, *others = colors
    fig, ax = lines.plot(data=weekly, gaps=True, x="timestamp", y=first,
                         color="lightgray", linewidth=1)
    for category in others:
        lines.plot_on(LinePlot(settings), data=weekly, gaps=True, x="timestamp",
                      y=category, color="lightgray", linewidth=1)
    for category, color in colors.items():
        lines.plot_on(LinePlot(settings), data=weekly_trend, gaps=True,
                      x="timestamp", y=f"{category}_rolling_avg", color=color,
                      linewidth=2.5, label=f"{category} ({SMOOTH_WEEKS}-week average)")
    for label, day in STORY_EVENTS.items():
        lines.plot_on(EventMarker(settings), date=day, label=label)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))

    # Every weekly share is a share of that week's messages: say how many that is,
    # and how many mention each topic, so a reader can judge how solid a week is.
    weekly_n = frame.set_index("timestamp").resample("W").size()
    mentions = " · ".join(f"{c}: {int(frame[c].sum()):,}" for c in colors)
    fig.text(
        0.5, -0.08,
        f"Based on {len(frame):,} messages over {len(weekly_n)} weeks "
        f"(median {int(weekly_n.median())} per week, fewest {int(weekly_n.min())}). "
        f"Messages mentioning each topic: {mentions}.",
        ha="center", fontsize=9, color="dimgray",
    )
    return fig, ax, weekly, weekly_trend


fig, ax, weekly, weekly_trend = topic_trend_plot(
    own_topics, FOCUS_COLORS,
    "Baby is taking over",
)

In [ ]:
# The running category from roos_01.4, pattern unchanged.
RUNNING_PATTERN = (
    r"(?i)("
    r"\brun(ning)?\b|courir|\btrail\b|\bbornes?\b|kilom[eè]tre|\bkm\b|marathon|"
    r"hardlopen|hardloopschoenen|joggen|strava|parkrun|\b10k\b|\btempo\b|"
    r"chaussures|\bshoes\b"
    r")"
)

own_sports = (
    Pipeline()
    .add(
        RegexFeature,
        name="running_words",
        column="message",
        pattern=RUNNING_PATTERN,
        feature="n_running_words",
        mode="count",
    )
    .apply(own_topics)
)
own_sports["running"] = own_sports["n_running_words"].gt(0).astype(float)

# sports = climbing OR running. A message that mentions both still counts once.
own_sports["sports"] = (own_sports["climbing"].gt(0) | own_sports["running"].gt(0)).astype(float)

SPORTS_COLORS = {"child": "#0072B2", "sports": "#D55E00"}

fig, ax, weekly_sports, weekly_sports_trend = topic_trend_plot(
    own_sports, SPORTS_COLORS,
    "Baby is taking over",
)

own_sports[["climbing", "running", "sports"]].sum().astype(int).rename("messages mentioning it")

In [ ]:
class EventPeriod(TransformBase):
    """Label each message with the last event that happened before it.

    The events split the timeline into periods: everything before the first event is
    "before <first event>", everything after event N is "after <event N>".
    """

    def transform(
        self,
        data: pd.DataFrame,
        column: str,
        events: dict[str, str],
        feature: str = "period",
    ) -> pd.DataFrame:
        days = pd.Series(pd.to_datetime(list(events.values())), index=list(events)).sort_values()
        ts = pd.to_datetime(data[column])
        if ts.dt.tz is not None:
            ts = ts.dt.tz_localize(None)
        edges = [pd.Timestamp("1970-01-01"), *days, pd.Timestamp("2100-01-01")]
        names = [f"before {days.index[0]}", *(f"after {label}" for label in days.index)]
        data[feature] = pd.cut(ts, bins=edges, labels=names, right=False)
        return data


by_period = Pipeline().add(EventPeriod, column="timestamp", events=my_events).apply(own_topics)

period_share = (
    by_period.groupby("period", observed=True)[list(FOCUS_COLORS)]
    .mean()
    .reset_index()
    .melt(id_vars="period", var_name="category", value_name="share")
)

period_plot = PlotSettings(
    figsize=(12, 5),
    title="Share of messages mentioning child / climbing, per period between my events",
    xlabel="",
    ylabel="share of the period's messages",
    xtick_rotation=15,
)
BarPlot(period_plot).plot(data=period_share, x="period", y="share",
                          hue="category", palette=FOCUS_COLORS)

# Sample size per period: a bar built on a few hundred messages is not a bar built on ten thousand.
by_period.groupby("period", observed=True).size().rename("messages in period")

## 3.3.6 Your turn, for real

Pick the event you care most about and the impact row from the table that matches your
prediction, then build that plot. The pieces you already have:

- **`FlagDates`** — any set of days becomes a boolean column to `Filter` or group on.
- **`CountValues(normalize=True)` / `Share(by=...)`** — shapes instead of volumes.
- **`SubtractBaseline`** — the residual move, whenever "compared to normal" is the claim.
- **`RegexFeature`** (lesson 1) — emoji, links, a word, a language marker, someone's
  catchphrase, as a column you can then compare before/after.
- **`FacetPlot`** — the same plot per author or per period, when "who" is the question.

This is also exactly what the `goad` MCP is for: ask your assistant to run
`goad_analysis_checklist` with your event and your predicted impact as the question, and
expect to be interviewed — the checklist will not let you skip past "what would change my
mind?". `goad_critique_visual` is waiting once you have a chart.

## 3.3.7 What to write down

1. **The event and the predicted impact — written before you plotted.** Quote yourself.
2. **The plot that could have falsified it**, and whether it did.
3. **One expected impact you could *not* find**, with the sample it would have needed to
   show up. A null result you can size is worth more than one you cannot.

## 3.3.8 Your event: when did the nausea start?

Pregnancy started **41 weeks before Benji's birth** (11 May 2025) — the standard due-date
convention counts weeks from the last period rather than conception, so "41 weeks before
birth" is used here as a rough proxy for "around the start of the pregnancy," not a
precise conception date.

Same rule as 3.3.2: write the prediction before looking. Nausea mentions are a **how**
row — a `RegexFeature` category, English/Dutch/French, built the same way as `child` and
`climbing` above, plotted as a weekly line with `EventMarker` on the positive test, the
calculated pregnancy-start date, and Benji's birth.

**My prediction:** I suppose these messages appear around 5 weeks after the start of the
pregnancy, and they disappear around week 16.

In [ ]:
# "nausea", English/Dutch/French, same shape as the child/climbing categories above.
NAUSEA_PATTERN = (
    r"(?i)("
    r"\bnausea(ted|ous)?\b|\bqueasy\b|"
    r"misselijk(heid)?|"
    r"naus[ée]e?(ux|se)?|mal au c[oœ]eur"
    r")"
)

own_nausea = (
    Pipeline()
    .add(
        RegexFeature,
        name="nausea_words",
        column="message",
        pattern=NAUSEA_PATTERN,
        feature="n_nausea_words",
        mode="count",
    )
    .apply(own_topics)
)
own_nausea["nausea"] = own_nausea["n_nausea_words"].gt(0).astype(float)

own_nausea[["nausea"]].sum().astype(int).rename("messages mentioning it")

In [ ]:
pregnancy_events = {
    "positive pregnancy test": my_events["Positive pregnancy test"],
    "pregnancy started (41 weeks before birth)": str(
        (pd.Timestamp(my_events["Benji is born"]) - pd.Timedelta(weeks=41)).date()
    ),
    "Benji is born": my_events["Benji is born"],
}
pregnancy_events

In [ ]:
# Turning the week-5 / week-16 prediction into calendar dates, off the same
# pregnancy-start anchor, so it can be drawn on the chart and checked by eye
# rather than taken on faith.
pregnancy_start = pd.Timestamp(pregnancy_events["pregnancy started (41 weeks before birth)"])

predicted_window = {
    "predicted nausea ends (week 16)": str((pregnancy_start + pd.Timedelta(weeks=16)).date()),
}
# Week 5 falls on 2024-09-01 -- the same date as "positive pregnancy test" above,
# so it's not drawn again here; that marker already sits where the prediction says
# nausea should start.
predicted_window

In [ ]:
NAUSEA_COLOR = "#CC79A7"

weekly_nausea = (
    own_nausea.set_index("timestamp")[["nausea"]]
    .resample("W")
    .mean()
    .reset_index()
)
nausea_trend = Pipeline().add(
    RollingAvg, column="nausea", window=SMOOTH_WEEKS, rename=True
).apply(weekly_nausea)

nausea_settings = PlotSettings(
    figsize=(12, 5),
    title="Nausea mentions per week",
    xlabel="",
    ylabel="share of weekly messages",
    xtick_rotation=45,
)
lines = LinePlot(nausea_settings)
fig, ax = lines.plot(data=weekly_nausea, gaps=True, x="timestamp", y="nausea",
                     color="lightgray", linewidth=1, label="weekly")
lines.plot_on(LinePlot(nausea_settings), data=nausea_trend, gaps=True, x="timestamp",
              y="nausea_rolling_avg", color=NAUSEA_COLOR, linewidth=2.5,
              label=f"{SMOOTH_WEEKS}-week average")
for label, day in pregnancy_events.items():
    lines.plot_on(EventMarker(nausea_settings), date=day, label=label)
# The predicted window, in purple so it reads as "my guess" against the grey
# "confirmed timeline" markers above -- a stand-in for the missing week-5 marker
# sits right on "positive pregnancy test" already, see the note in the cell above.
for label, day in predicted_window.items():
    lines.plot_on(EventMarker(nausea_settings), date=day, label=label,
                  color="purple", text_offset_days=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))

weekly_n = own_nausea.set_index("timestamp").resample("W").size()
fig.text(
    0.5, -0.08,
    f"Based on {len(own_nausea):,} messages over {len(weekly_n)} weeks "
    f"(median {int(weekly_n.median())} per week, fewest {int(weekly_n.min())}). "
    f"Messages mentioning nausea: {int(own_nausea['nausea'].sum()):,}.",
    ha="center", fontsize=9, color="dimgray",
)

Read this against your prediction above, not for a headline number — a handful of
`RegexFeature` hits spread across a week is a thin signal (check `weekly_n` for the weeks
right around the computed pregnancy-start marker before trusting any rise you see there).

**What I found:** (write it here — did the rise line up with the calculated
pregnancy-start marker, before it, after it, or not at all?)

In [ ]:
pregnancy_end = pd.Timestamp(pregnancy_events["Benji is born"])
PREGNANCY_WEEKS = 41

# own_nausea["timestamp"] is tz-aware (UTC); pregnancy_start/end are naive -- same fix
# EventPeriod uses above, drop the tz for this comparison rather than localise the
# milestone dates to a timezone they were never given.
roos_pregnancy = own_nausea.copy()
roos_pregnancy["timestamp"] = roos_pregnancy["timestamp"].dt.tz_localize(None)
roos_pregnancy = roos_pregnancy[
    (roos_pregnancy["author"] == "Roos")
    & (roos_pregnancy["timestamp"] >= pregnancy_start)
    & (roos_pregnancy["timestamp"] <= pregnancy_end)
]

# Week 1 = the first week after pregnancy_start -- a week-of-pregnancy count,
# not a calendar week.
roos_pregnancy["pregnancy_week"] = (
    (roos_pregnancy["timestamp"] - pregnancy_start).dt.days // 7
) + 1

# "Mentions" = total regex hits (n_nausea_words), so a message that says "misselijk"
# twice counts twice -- not just whether the message mentions it at all.
by_week = (
    roos_pregnancy.groupby("pregnancy_week")["n_nausea_words"]
    .sum()
    .reindex(range(1, PREGNANCY_WEEKS + 1), fill_value=0)
    .rename("mentions")
    .reset_index()
)

week_settings = PlotSettings(
    figsize=(12, 5),
    title="Roos's nausea mentions, by week of pregnancy",
    xlabel="week of pregnancy",
    ylabel="mentions (count)",
)
lines_week = LinePlot(week_settings)
fig, ax = lines_week.plot(data=by_week, x="pregnancy_week", y="mentions",
                          marker="o", color=NAUSEA_COLOR)

fig.text(
    0.5, -0.08,
    f"Roos's own messages only, within the pregnancy window "
    f"({len(roos_pregnancy):,} messages, {int(roos_pregnancy['n_nausea_words'].sum())} "
    f"total nausea mentions).",
    ha="center", fontsize=9, color="dimgray",
)

In [ ]:
# "fatigue", English/Dutch/French, same shape as the nausea category above.
# \bmoe\b is boundary-matched to avoid "moeder"/"moeite"; see the note above about
# what that trade-off misses (e.g. "doodmoe").
FATIGUE_PATTERN = (
    r"(?i)("
    r"\bfatigue(d)?\b|\btired\b|\bexhaust(ed|ion)\b|\bworn out\b|"
    r"\bmoe\b|\bmoeheid\b|\bvermoeidheid\b|\bvermoeid\b|\buitgeput\b|"
    r"fatigu[ée]e?|[ée]puis[ée]e?|crev[ée]e?"
    r")"
)

own_fatigue = (
    Pipeline()
    .add(
        RegexFeature,
        name="fatigue_words",
        column="message",
        pattern=FATIGUE_PATTERN,
        feature="n_fatigue_words",
        mode="count",
    )
    .apply(own_topics)
)
own_fatigue["fatigue"] = own_fatigue["n_fatigue_words"].gt(0).astype(float)

own_fatigue[["fatigue"]].sum().astype(int).rename("messages mentioning it")

In [ ]:
FATIGUE_COLOR = "#E69F00"

# Same tz fix as the nausea-by-week cell: own_fatigue["timestamp"] is tz-aware (UTC),
# pregnancy_start/end are naive.
roos_fatigue = own_fatigue.copy()
roos_fatigue["timestamp"] = roos_fatigue["timestamp"].dt.tz_localize(None)
roos_fatigue = roos_fatigue[
    (roos_fatigue["author"] == "Roos")
    & (roos_fatigue["timestamp"] >= pregnancy_start)
    & (roos_fatigue["timestamp"] <= pregnancy_end)
]
roos_fatigue["pregnancy_week"] = (
    (roos_fatigue["timestamp"] - pregnancy_start).dt.days // 7
) + 1

fatigue_by_week = (
    roos_fatigue.groupby("pregnancy_week")["n_fatigue_words"]
    .sum()
    .reindex(range(1, PREGNANCY_WEEKS + 1), fill_value=0)
    .rename("mentions")
    .reset_index()
)

fatigue_settings = PlotSettings(
    figsize=(12, 5),
    title="Roos's fatigue mentions, rising and declining across the pregnancy",
    xlabel="week of pregnancy",
    ylabel="mentions (count)",
    highlight_color=FATIGUE_COLOR,
    base_color="lightgrey",
)
fig, ax = RiseDeclinePlot(fatigue_settings).plot(
    data=fatigue_by_week, x="pregnancy_week", y="mentions",
    rise_label="rising week-on-week", decline_label="flat or declining",
)

fig.text(
    0.5, -0.08,
    f"Roos's own messages only, within the pregnancy window "
    f"({len(roos_fatigue):,} messages, {int(roos_fatigue['n_fatigue_words'].sum())} "
    f"total fatigue mentions).",
    ha="center", fontsize=9, color="dimgray",
)

## 3.3.10 A second symptom: fatigue, rising and declining across the pregnancy

Same scope as 3.3.9 — Roos's own messages only, the pregnancy window only, weekly
counts, week-of-pregnancy on the x-axis — but a new category (`fatigue`,
English/Dutch/French) and a new plot class: `RiseDeclinePlot`, added to
`scripts/plots.py` because neither `LinePlot` nor `BarPlot` there say anything about
*direction* — colouring each week-to-week segment by whether the count went up or down,
rather than plotting one flat colour and leaving direction to be read off the slope by
eye.

**Word list, for review** (categories are yours to defend, same rule as `child`/
`climbing`/`nausea`): English `fatigue`/`tired`/`exhausted`/`exhaustion`/`worn out`;
Dutch `moe`/`moeheid`/`vermoeid`/`vermoeidheid`/`uitgeput`; French
`fatigue`/`fatigué(e)`/`épuisé(e)`/`crevé(e)`. `moe` is boundary-matched so it doesn't
catch `moeder`/`moeite`, but that also means a compound like `doodmoe` (very common
Dutch for "dead tired") is missed — worth extending if it matters to you.

## 3.3.9 Zooming in: Roos's nausea mentions, week by week of the pregnancy

Same `nausea` category as 3.3.8, re-scoped to a narrower question:

- **Only the pregnancy window** — `pregnancy_start` to Benji's birth, nothing before or
  after.
- **Roos's own messages only** — `own_nausea` filtered to `author == "Roos"`.
- **Raw mention counts on the y-axis**, not a weekly share — a straight count of how
  many times "nausea" (in any of the three languages) shows up.
- **Week-of-pregnancy on the x-axis**, not calendar dates — week 1 is the first week
  after `pregnancy_start`, counting up to week 41.

Worth naming before looking: this was already thin at n=38 for *both* of you across the
*whole* dataset. Restricting to Roos-only, pregnancy-only cuts that further — read any
single week's bar as one data point, not a trend, unless several adjacent weeks agree.